In [1]:
import pandas as pd

# Read your CSV file directly
df = pd.read_csv("../tabular_3/diabetes_3.csv")

print(" File loaded successfully!")
print(f"Dataset shape: {df.shape}")


 File loaded successfully!
Dataset shape: (768, 9)


In [2]:
# Show all column names
print("Column names:")
print(df.columns.tolist())

# check data types
print("\nDtypes:")
print(df.dtypes)

# Preview first 5 rows
df.head()

Column names:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

Dtypes:
Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
import numpy as np
from pathlib import Path

# Define feature template
TEMPLATE = ['Age', 'Sex', 'BMI', 'GenHlth', 'HighBP', 'DiffWalk', 'HighChol', 'HeartDiseaseorAttack']

# Make a copy of the raw dataframe
df_raw = df.copy()

# Initialize aligned dataframe and mask dataframe
aligned = pd.DataFrame(index=df_raw.index, columns=TEMPLATE, dtype='float64')
mask = pd.DataFrame(0, index=df_raw.index, columns=[f"{col}_mask" for col in TEMPLATE], dtype='int8')


# 1. Directly mapped features
for c in ['Age', 'BMI']:
    if c in df_raw.columns:
        aligned[c] = df_raw[c]
        mask[f"{c}_mask"] = 1


# 2. Sex (Pima dataset is female only)
aligned['Sex'] = 0
mask['Sex_mask'] = 0  # derived


# 3. Derive HighBP from BloodPressure (>=90 → 1, else 0)
if 'BloodPressure' in df_raw.columns:
    bp = pd.to_numeric(df_raw['BloodPressure'], errors='coerce')
    aligned['HighBP'] = (bp >= 90).astype('int8')
    mask['HighBP_mask'] = 0  # derived
else:
    aligned['HighBP'] = 0
    mask['HighBP_mask'] = 0


# 4. Derive 5-level GenHlth based on BMI + BP + Glucose
def _score_bmi(series):
    x = pd.to_numeric(series, errors='coerce')
    return np.where(x >= 30, 2, np.where(x >= 25, 1, 0))

def _score_bp(series):
    x = pd.to_numeric(series, errors='coerce')
    return np.where(x >= 90, 2, np.where(x >= 80, 1, 0))

def _score_glucose(series):
    x = pd.to_numeric(series, errors='coerce')
    return np.where(x >= 140, 2, np.where(x >= 100, 1, 0))

if set(['BMI', 'BloodPressure', 'Glucose']).issubset(df_raw.columns):
    bmi_s = _score_bmi(df_raw['BMI'])
    bp_s  = _score_bp(df_raw['BloodPressure'])
    glu_s = _score_glucose(df_raw['Glucose'])

    total = bmi_s + bp_s + glu_s  # range 0–6

    genhlth_5 = np.select(
        [
            total <= 1,
            total == 2,
            (total >= 3) & (total <= 4),
            total == 5,
            total >= 6
        ],
        [1, 2, 3, 4, 5],
        default=3
    ).astype('int8')

    aligned['GenHlth'] = genhlth_5
    mask['GenHlth_mask'] = 0
else:
    aligned['GenHlth'] = 3
    mask['GenHlth_mask'] = 0


# 5. Fill missing features (DiffWalk, HighChol, HeartDiseaseorAttack)
for c in ['DiffWalk', 'HighChol', 'HeartDiseaseorAttack']:
    aligned[c] = 0
    mask[f"{c}_mask"] = 0


# 6. Add label column
if 'Outcome' in df_raw.columns:
    label = df_raw['Outcome'].astype('int8')
else:
    label = pd.Series(np.nan, index=df_raw.index, name='label')


# 7. Combine and save with mask
aligned_with_mask = pd.concat([aligned, mask, label.rename('label')], axis=1)

out_dir = Path("../tabular_3")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "tabular3_aligned_with_mask.csv"

aligned_with_mask.to_csv(out_path, index=False)

print(f"\n Saved aligned dataset with mask to: {out_path}")
print("Shape:", aligned_with_mask.shape)
aligned_with_mask.head()



 Saved aligned dataset with mask to: ../tabular_3/tabular3_aligned_with_mask.csv
Shape: (768, 17)


,Age,Sex,BMI,GenHlth,HighBP,DiffWalk,HighChol,HeartDiseaseorAttack,Age_mask,Sex_mask,BMI_mask,GenHlth_mask,HighBP_mask,DiffWalk_mask,HighChol_mask,HeartDiseaseorAttack_mask,label
0,50,0,33.6,3,0,0,0,0,1,0,1,0,0,0,0,0,1
1,31,0,26.6,1,0,0,0,0,1,0,1,0,0,0,0,0,0
2,32,0,23.3,2,0,0,0,0,1,0,1,0,0,0,0,0,1
3,21,0,28.1,1,0,0,0,0,1,0,1,0,0,0,0,0,0
4,33,0,43.1,3,0,0,0,0,1,0,1,0,0,0,0,0,1


In [4]:
# Split 6:2:2 with stratification on label, keep masks
from pathlib import Path
from sklearn.model_selection import train_test_split
import pandas as pd

in_path  = Path("../tabular_3/tabular3_aligned_with_mask.csv")
out_dir  = Path("../tabular_3")
out_dir.mkdir(parents=True, exist_ok=True)

# Load the aligned+mask dataset (17 columns: 8 feats + 8 masks + label)
df_all = pd.read_csv(in_path)

# Safety: ensure label column exists & is int
assert 'label' in df_all.columns, "Missing 'label' column."
df_all['label'] = df_all['label'].astype('int8')

# 6 : 2 : 2 split  (train : val : test)
train_ratio, val_ratio, test_ratio = 0.6, 0.2, 0.2

# 1) train vs (val+test)
df_train, df_temp = train_test_split(
    df_all,
    test_size=(1 - train_ratio),
    random_state=42,
    stratify=df_all['label']
)

# 2) val vs test (relative)
rel = val_ratio / (val_ratio + test_ratio)  # 0.5 if 0.2/0.4, here 0.5
df_val, df_test = train_test_split(
    df_temp,
    test_size=(1 - rel),
    random_state=42,
    stratify=df_temp['label']
)

# Save three splits (same folder as tabular_3)
train_path = out_dir / "diabetes_3_train.csv"
val_path   = out_dir / "diabetes_3_val.csv"
test_path  = out_dir / "diabetes_3_test.csv"

df_train.to_csv(train_path, index=False)
df_val.to_csv(val_path, index=False)
df_test.to_csv(test_path, index=False)

print(" Saved train/val/test splits successfully:")
print(f" - Train: {train_path}  shape={df_train.shape}")
print(f" - Val:   {val_path}    shape={df_val.shape}")
print(f" - Test:  {test_path}   shape={df_test.shape}")

# Quick label balance check
def label_ratio(df):
    return df['label'].value_counts(normalize=True).round(3).to_dict()

print("\nLabel distribution:")
print("Train:", label_ratio(df_train))
print("Val:  ", label_ratio(df_val))
print("Test: ", label_ratio(df_test))


 Saved train/val/test splits successfully:
 - Train: ../tabular_3/diabetes_3_train.csv  shape=(460, 17)
 - Val:   ../tabular_3/diabetes_3_val.csv    shape=(154, 17)
 - Test:  ../tabular_3/diabetes_3_test.csv   shape=(154, 17)

Label distribution:
Train: {0: 0.65, 1: 0.35}
Val:   {0: 0.649, 1: 0.351}
Test:  {0: 0.656, 1: 0.344}
